In [ ]:
def compute_grpo_advantage(rewards: torch.Tensor, num_generations: int):
    """
    GRPO 优势计算：组内标准化
    
    rewards: [batch_size * G]
    returns: advantages: [batch_size * G]
    """
    batch_size = rewards.size(0) // num_generations
    rewards_grouped = rewards.view(batch_size, num_generations)
    
    mean = rewards_grouped.mean(dim=1, keepdim=True)
    std = rewards_grouped.std(dim=1, keepdim=True) + 1e-8
    
    advantages = (rewards_grouped - mean) / std
    return advantages.view(-1)

In [ ]:
def compute_grpo_loss(
    log_probs: torch.Tensor,
    ref_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    beta: float = 0.04,
    epsilon: float = 0.2
):
    """
    GRPO 损失计算
    
    ratio = exp(log_probs - old_log_probs)
    clipped_ratio = clip(ratio, 1-eps, 1+eps)
    loss = -min(ratio * A, clipped_ratio * A) + beta * KL
    """
    # 重要性采样
    ratio = torch.exp(log_probs - old_log_probs)

    # PPO裁剪
    clipped_ratio = torch.clamp(ratio,1-epsilon,1+epsilon)
    policy_loss = -torch.min(ratio * advantages ,clipped_ratio * advantages  )

    # KL惩罚
    kl  = torch.exp(ref_log_probs -log_log_probs ) - (ref_log_probs - log_probs)-1
    return policy_loss + beta * kl